# 08 — Regression Discontinuity Designs
**References:** Thistlethwaite & Campbell (1960) · Imbens & Lemieux (2008, *J. Econometrics*) · Lee & Lemieux (2010, *JEL*) · Cattaneo, Idrobo & Titiunik (2020) · McCrary (2008)

## Narrative thread
```
Threshold rules -> Identification at the cutoff -> Local linear estimation -> Bandwidth choice -> Fuzzy RD -> Validity checks
```

## The design

A **running variable** $X$ (test score, poverty index, vote share) assigns treatment by a
hard threshold: $W = \mathbb{1}\{X \geq c\}$. Units *just* above and *just* below the
cutoff are comparable in everything except treatment — a local natural experiment.

**Identification (continuity assumption):** if $E[Y(0) \mid X = x]$ and
$E[Y(1) \mid X = x]$ are continuous at $c$, then

$$\tau_{SRD} = \lim_{x \downarrow c} E[Y \mid X = x] - \lim_{x \uparrow c} E[Y \mid X = x]
= E[Y(1) - Y(0) \mid X = c]$$

The estimand is the effect **at the cutoff** — the most local of local effects, but with
near-experimental credibility (Lee & Lemieux: "the closest thing to an RCT found in
observational settings").

## Estimation: local linear regression

Fit separate linear regressions within a **bandwidth** $h$ on each side of the cutoff
(triangular kernel weights are standard). Global high-order polynomials are **discouraged**
(Gelman & Imbens 2019): they put weight on far-away observations and produce erratic
cutoff estimates. Bandwidth is bias-variance: small $h$ → less bias, more noise. Modern
practice uses the MSE-optimal bandwidth with robust bias-corrected inference (Calonico,
Cattaneo & Titiunik 2014).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [ ]:
# ── Sharp RD: scholarship at a test-score cutoff ──────────────────────────
n = 4000
c = 60.0                                            # cutoff
score = np.random.uniform(20, 100, n)               # running variable
w = (score >= c).astype(int)                        # sharp assignment
# Smooth potential-outcome curves + a true jump of 8 at the cutoff
y = 10 + 0.45*score - 0.002*(score - 60)**2 + 8*w + np.random.normal(0, 5, n)
df = pd.DataFrame(dict(score=score, w=w, y=y))

def local_linear_rd(df, c, h):
    """Triangular-kernel local linear fit on each side; returns jump estimate."""
    d = df[np.abs(df.score - c) <= h].copy()
    d['xc'] = d.score - c
    d['k']  = 1 - np.abs(d.xc) / h                  # triangular kernel
    m = smf.wls('y ~ xc * w', d, weights=d.k).fit()
    return m.params['w'], m.bse['w']

tau_hat, se = local_linear_rd(df, c, h=15)
print(f'True effect at cutoff: 8.00')
print(f'Local linear RD (h=15): {tau_hat:.2f}  (SE {se:.2f})')

# Binned scatter + fitted lines (the canonical RD plot)
bins = np.linspace(20, 100, 41)
df['bin'] = pd.cut(df.score, bins)
bs = df.groupby('bin', observed=True).agg(x=('score', 'mean'), y=('y', 'mean'))

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(bs.x, bs.y, s=22, color='#444', zorder=3, label='binned means')
for side, color in [(df.score < c, '#1f77b4'), (df.score >= c, '#d62728')]:
    d = df[side & (np.abs(df.score - c) <= 15)]
    coef = np.polyfit(d.score, d.y, 1)
    xs = np.linspace(d.score.min(), d.score.max(), 50)
    ax.plot(xs, np.polyval(coef, xs), color=color, lw=2)
ax.axvline(c, color='k', ls='--', lw=1)
ax.set_xlabel('test score (running variable)'); ax.set_ylabel('outcome')
ax.set_title(f'Sharp RD: estimated jump at cutoff = {tau_hat:.2f}')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── Bandwidth sensitivity: the essential robustness plot ─────────────────
hs = np.arange(4, 41, 2)
est = np.array([local_linear_rd(df, c, h) for h in hs])

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.errorbar(hs, est[:, 0], yerr=1.96*est[:, 1], fmt='o-', ms=4, capsize=3, color='#1f77b4')
ax.axhline(8, color='#d62728', ls='--', lw=1.5, label='true effect')
ax.set_xlabel('bandwidth h'); ax.set_ylabel('RD estimate')
ax.set_title('Estimate vs bandwidth: stable across reasonable h = credible')
ax.legend(); plt.tight_layout(); plt.show()
print('Small h: noisy but unbiased. Large h: precise but curvature bias creeps in.')

## Fuzzy RD

When crossing the cutoff only *changes the probability* of treatment (imperfect
compliance), the design is **fuzzy**: the cutoff indicator is an **instrument** for
treatment (this is IV at a threshold — notebook 07):

$$\tau_{FRD} = \frac{\lim_{x\downarrow c} E[Y|X{=}x] - \lim_{x\uparrow c} E[Y|X{=}x]}
{\lim_{x\downarrow c} E[W|X{=}x] - \lim_{x\uparrow c} E[W|X{=}x]}$$

— the jump in outcomes divided by the jump in take-up, a LATE for cutoff compliers.

## Validity checks

1. **Manipulation / density test (McCrary 2008):** if units can sort across the cutoff
   (e.g., teachers nudge scores), the density of $X$ jumps at $c$. Test for continuity of
   the density.
2. **Covariate continuity:** predetermined covariates must NOT jump at the cutoff.
3. **Placebo cutoffs:** no significant jumps at fake thresholds.


In [ ]:
# ── Fuzzy RD + McCrary-style manipulation check ───────────────────────────
# Fuzzy: crossing the cutoff raises P(treatment) from 20% to 75%
p_take = np.where(score >= c, 0.75, 0.20)
w_fuzzy = np.random.binomial(1, p_take)
y_fuzzy = 10 + 0.45*score + 8*w_fuzzy + np.random.normal(0, 5, n)
dff = pd.DataFrame(dict(score=score, w=w_fuzzy, y=y_fuzzy, above=(score >= c).astype(int)))

h = 15
d = dff[np.abs(dff.score - c) <= h].copy()
d['xc'] = d.score - c; d['k'] = 1 - np.abs(d.xc)/h
jump_y = smf.wls('y ~ xc * above', d, weights=d.k).fit().params['above']
jump_w = smf.wls('w ~ xc * above', d, weights=d.k).fit().params['above']
print(f'Jump in outcome at cutoff:  {jump_y:.2f}')
print(f'Jump in take-up at cutoff:  {jump_w:.2f}')
print(f'Fuzzy RD estimate:          {jump_y / jump_w:.2f}   (true effect: 8.00)')

# Manipulation: simulate sorting (units just below sneak above), check density
score_manip = score.copy()
sneak = (score > c - 3) & (score < c) & (np.random.rand(n) < 0.6)
score_manip[sneak] = c + np.random.uniform(0, 2, sneak.sum())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, s, title in [(axes[0], score, 'No manipulation: smooth density'),
                     (axes[1], score_manip, 'Manipulation: density jumps at cutoff')]:
    ax.hist(s, bins=60, color='#1f77b4', alpha=0.85)
    ax.axvline(c, color='#d62728', ls='--')
    ax.set_title(title); ax.set_xlabel('running variable')
plt.tight_layout(); plt.show()

## Key takeaways

| Concept | Statement |
|---|---|
| Sharp RD | Threshold rule + continuity ⟹ effect at the cutoff identified |
| Estimand | $E[Y(1)-Y(0) \mid X=c]$ — local to the cutoff population |
| Estimation | Local linear with triangular kernel; avoid global polynomials |
| Bandwidth | Bias-variance dial; show a sensitivity plot; use CCT-robust CIs |
| Fuzzy RD | Jump in $Y$ ÷ jump in take-up = IV at the threshold |
| Validity | McCrary density test, covariate continuity, placebo cutoffs |
